In [25]:
# Prompt taken from MIND paper

BASELINE_PROMPT = """
Given the meme, with the
Text: "{transcription}" embedded in the image, your
task is to determine whether this meme is harm-
ful or not, in order to maintain the harmlessness
and integrity of information on the Internet. Please
leverage your extensive knowledge to deeply an-
alyze and understand this meme, and give your
final judgment. Your output should stricly follow
the format: "Thought: [Your analysis] Answer:
[harmful/harmless].
"""

In [41]:
from dotenv import load_dotenv
from huggingface_hub import InferenceClient
from termcolor import colored
import json
import os

load_dotenv()

client = InferenceClient(api_key=os.environ["HF_TOKEN"])

# MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct:together"

c = 0

def lookup_drive_id(image_name, lookup_path):
    with open(lookup_path, "r") as f:
        data = json.load(f)

    for item in data:
        if item["name"] == image_name:
            return item["id"]

    raise ValueError(f"Image '{image_name}' not found.")


def make_drive_download_url(file_id):
    return f"https://drive.google.com/uc?export=download&id={file_id}"


def get_model_res(model_name, prompt, image_id, current_path):
    global c
    try:
        drive_id = lookup_drive_id(
            image_id,
            f"{current_path}/drive_lookup.json",
        )
        image_url = make_drive_download_url(drive_id)
    except ValueError as e:
        print(colored(f"{e} Skipping count: {c}", "red"))
        c += 1
        return ""

    for attempt in range(3):
        try:
            completion = client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {
                                "type": "image_url",
                                "image_url": {"url": image_url},
                            },
                        ],
                    }
                ],
                temperature=0,
                top_p=None,
                # max_tokens=1024,
            )
            output = completion.choices[0].message.content.strip()
            
            pred_label = output.split("Answer: ")[1]

            if pred_label not in ["harmful", "harmless"]:
                continue

            return output, pred_label

        except Exception as e:
            print(colored(f"Attempt {attempt + 1}/3 failed: {e}", "red"))
            if attempt == 2:
                raise

In [ ]:
import pandas as pd
import json

DATASET = "../dataset/misogyny/tamil"
MODEL_NAME = "google/gemma-4-31B-it:fastest"

path = DATASET.split("/")
dataset_name, language = path[-2], path[-1] 
output_path = f"{dataset_name}_{language}.jsonl"

df = pd.read_csv(f"{DATASET}/test_with_labels.csv")

results = []

with open(output_path, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        image_id = str(row["image_id"])
        transcription = "" if pd.isna(row["transcriptions"]) else str(row["transcriptions"])

        prompt = BASELINE_PROMPT.format(transcription=transcription)

        try:
            response, pred_label = get_model_res(
                model_name=MODEL_NAME,
                prompt=prompt,
                image_id=image_id,
                current_path=DATASET,
            )
        except Exception as e:
            print(f"Error processing {image_id}: {e}")
            response = ""

        result = {
            "image_id": image_id,
            "label": row["labels"],
            "response": response,
            "pred_label": pred_label 
        }
        
        f.write(json.dumps(result, ensure_ascii=False) + "\n")
        f.flush()  # ensures immediate write to disk

        print(f"Finished {image_id}")


Finished 522
Finished 738
Finished 741
Finished 661
Finished 412
Finished 679
Finished 627
Finished 514
Finished 860
Finished 137
Finished 812
Finished 77
Finished 637
Finished 974
Finished 939
Finished 900
Finished 281
Finished 884
Finished 762
Finished 320
Finished 550
Finished 175
Finished 372
Finished 528
Finished 211
Finished 236
Finished 102
Finished 987
Finished 903
Finished 948
Finished 347
Finished 140
Finished 622
Finished 500
Finished 371
Finished 199
Finished 688
Finished 585
Finished 902
Finished 60
Finished 329
Finished 97
Finished 313
Finished 975
Finished 300
Finished 278
Finished 925
Finished 602
Finished 440
Finished 838
Finished 571
Finished 880
Finished 262
Finished 579
Finished 24
Finished 31
Finished 618
Finished 11
Finished 222
Finished 821
Finished 297
Finished 55
Finished 543
Finished 210
Finished 605
Finished 693
Finished 663
Finished 867
Finished 71
Finished 544
Finished 108
Finished 494
Finished 591
Finished 742
Finished 293
Attempt 1/3 failed: Server error 

In [43]:
!pip install scikit-learn


 _     _  _______  _  ______   _______  ______   _______  _     _  _______  _______  _       
(_)   (_)(_______)| |(______) (_______)(_____ \ (_______)(_)   | |(_______)(_______)(_)      
 _     _  _     _ | | _     _  _____    _____) ) _______  _____| |    _     _______  _       
| |   | || |   | || || |   | ||  ___)  |  __  / |  ___  ||  _   _)   | |   |  ___  || |      
 \ \ / / | |___| || || |__/ / | |      | |  \ \ | |   | || |  \ \    | |   | |   | || |_____ 
  \___/   \_____/ |_||_____/  |_|      |_|   |_||_|   |_||_|   \_)   |_|   |_|   |_||_______)
                                                                                             
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.0 MB/s eta 0:00:00
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 11.0 MB/s eta 0:00:00a 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl 

In [45]:
import json
from sklearn.metrics import accuracy_score, f1_score

y_true = []
y_pred = []

with open("misogyny_malayalam.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        label = record["label"].lower()
        if label == "misogyny":
            label = "harmful"
        else:
            label = "harmless"
        y_true.append(label)
        y_pred.append(record["pred_label"])

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print(f"Accuracy : {accuracy:.4f}")
print(f"Macro F1 : {macro_f1:.4f}")

Accuracy : 0.8400
Macro F1 : 0.8217
